# Setting things up

## About this notebook

``Discretizer`` is the first step of every **Carver**: it turns each feature into a few ordered modalities, each frequent enough, before any combination of modalities is searched. It can also be used on its own.

In this notebook, we run it on the Adult census data and look at what it does to four kinds of features: continuous, discrete, categorical and ordinal.

## Installation

In [1]:
# %pip install "AutoCarver[jupyter]"

## Census Data

In this example notebook, we will use the Adult dataset. It comes from the 1994 U.S. Census database. Each row is a person, and the target is whether their yearly income is above 50,000 dollars (binary classification).

We load the copy hosted on OpenML through scikit-learn. In this copy, missing values (marked ``?`` in the original files) are already ``NaN``.

In [2]:
from sklearn.datasets import fetch_openml

# Adult dataset from OpenML
adult_data = fetch_openml("adult", version=2, as_frame=True).frame

# Display the first few rows of the dataset
adult_data.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,class
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,NaN,103497,Some-college,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States,<=50K


## Target type

In [3]:
target = "income"

# conversion to 0/1
adult_data[target] = (adult_data.pop("class") == ">50K").astype(int)

# target rate
adult_data[target].value_counts(dropna=False)

income
0    37155
1    11687
Name: count, dtype: int64

The target ``"income"`` is a binary target used in a classification task.

## Data Sampling

In [4]:
from sklearn.model_selection import train_test_split

# stratified sampling by target
train_set, dev_set = train_test_split(adult_data, test_size=0.20, random_state=42, stratify=adult_data[target])

# checking target rate per dataset
train_set[target].mean(), dev_set[target].mean()

(np.float64(0.23927008420136667), np.float64(0.23932848807452145))

## Picking up columns to Discretize

In [5]:
train_set.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
34342,71,Private,77253,HS-grad,9,Never-married,Handlers-cleaners,Not-in-family,White,Male,0,0,17,United-States,0
18559,17,Private,329783,10th,6,Never-married,Sales,Other-relative,White,Female,0,0,10,United-States,0
12477,27,Private,91257,HS-grad,9,Married-civ-spouse,Other-service,Husband,White,Male,0,0,40,El-Salvador,0
560,43,Private,125577,HS-grad,9,Separated,Adm-clerical,Unmarried,Black,Female,0,0,40,United-States,0
3427,31,Private,137978,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,40,United-States,0


In [6]:
# column data types
train_set.dtypes

age                  int64
workclass         category
fnlwgt               int64
education         category
education-num        int64
marital-status    category
occupation        category
relationship      category
race              category
sex               category
capital-gain         int64
capital-loss         int64
hours-per-week       int64
native-country    category
income               int64
dtype: object

In [7]:
# share of missing values per column, when any
missing = train_set.isna().mean()
missing[missing > 0]

workclass         0.056919
occupation        0.057175
native-country    0.017429
dtype: float64

Three categorical features have missing values: ``workclass`` (5.7% of the train rows), ``occupation`` (5.7%) and ``native-country`` (1.7%).

* ``"education"`` is the only qualitative ordinal feature. It goes in ``ordinals``, with its values listed from the lowest to the highest degree.

* ``"sex"``, ``"marital-status"``, ``"occupation"``, ``"relationship"``, ``"race"``, ``"native-country"`` and ``"workclass"`` are categorical features with no order. They go in ``categoricals``.

* ``"capital-gain"`` and ``"capital-loss"`` are continuous, whilst ``"education-num"``, ``"age"`` and ``"hours-per-week"`` take a few dozen integer values. All five go in ``numericals``.

* ``"fnlwgt"`` is the census sampling weight: the number of people a row stands for in the U.S. population. It describes how the sample was drawn, not the person, so we leave it out.

In [8]:
from AutoCarver import Features

features = Features(
    categoricals=["sex", "workclass", "marital-status", "occupation", "relationship", "race", "native-country"],
    numericals=["capital-gain", "capital-loss", "education-num", "hours-per-week", "age"],
    ordinals={
        "education": [
            "Preschool", "1st-4th", "5th-6th", "7th-8th", "9th", "10th", "11th", "12th", "HS-grad",
            "Some-college", "Assoc-voc", "Assoc-acdm", "Bachelors", "Masters", "Prof-school", "Doctorate",
        ]
    },
)

# Using Discretizer
## Discretizer Settings
### Representativness of modalities

The attribute ``min_freq`` sets the minimum frequency per modality. It is used by **Discretizers**:

- For quantitative features, values more frequent than ``min_freq`` are kept as their own modality, and the other values are cut in quantiles of about ``min_freq`` each.

- For qualitative features, modalities significantly less frequent than ``min_freq`` are grouped, either with a default value (categorical features) or with a neighbouring modality (ordinal features).

**Carvers** call the same pipeline with ``min_freq/2``, so that the combination search starts from finer modalities.

In [9]:
min_freq = 0.02

**Tip:** set it between ``0.01`` (slower, less robust) and ``0.05`` (faster, more robust). The default is ``0.02``.

## Fitting Discretizer

* Quantitative features are discretized by ``QuantitativeDiscretizer``:
    1. ``ContinuousDiscretizer`` cuts each feature into quantiles and keeps over-represented values (more frequent than ``min_freq``) as their own modality
    2. ``OrdinalDiscretizer`` groups any remaining under-represented modality with a neighbouring one

* Qualitative features are discretized by ``QualitativeDiscretizer``:
    1. ``StringDiscretizer`` converts values to ``str`` when needed
    2. For ordinal features, ``OrdinalDiscretizer`` groups under-represented values with a neighbouring modality, in the user-provided order
    3. For categorical features, ``CategoricalDiscretizer`` groups under-represented values into a default value (``"__OTHER__"``)

Behavioral options (``copy``, ``verbose``, ``dropna``, ``ordinal_encoding`` …) go in a ``config`` dictionary, typed as ``ProcessingConfig``.

In [10]:
from AutoCarver.discretizers import Discretizer

# intiating Discretizer
discretizer = Discretizer(features=features, min_freq=min_freq, config={"copy": True, "verbose": False})

# fitting on training sample
train_set_processed = discretizer.fit_transform(train_set, train_set[target])

## Discretizer Analysis
### Quantitative Continuous Feature

In [11]:
# Discretization Summary
feature = "capital-gain"
discretizer.summary.loc[[str(discretizer.features[feature])]]

content
feature                   label                                     
Numerical('capital-gain') (-inf, 0.00e+00]          (-inf, 0.00e+00]
                          (0.00e+00, 3.41e+03]  (0.00e+00, 3.41e+03]
                          (3.41e+03, 7.30e+03]  (3.41e+03, 7.30e+03]
                          (7.30e+03, inf)            (7.30e+03, inf)

In [12]:
# Not discretized distribution
stats = train_set[feature].value_counts(dropna=True, normalize=True)
print(f"Over-represented values of {feature}:\n", stats[stats >= min_freq])

Over-represented values of capital-gain:
 capital-gain
0    0.918256
Name: proportion, dtype: float64


In [13]:
# Discretized distribution
disc_stats = train_set_processed[feature].value_counts(dropna=True, normalize=True)
print(f"Discretized distribution of {feature}:\n", disc_stats)

Discretized distribution of capital-gain:
 capital-gain
(-inf, 0.00e+00]        0.918256
(7.30e+03, inf)         0.034116
(3.41e+03, 7.30e+03]    0.026642
(0.00e+00, 3.41e+03]    0.020986
Name: proportion, dtype: float64


For quantitative continuous feature ``capital-gain``:

* An over-represented value has been identified and kept by itself: value ``0`` represents 91.8% of observed data (more than ``min_freq=0.02``)
* The remaining 8.2% of values have been cut into three bins, holding 2.1%, 2.7% and 3.4% of observed data

### Quantitative Discrete Feature

In [14]:
# Discretization Summary
feature = "hours-per-week"
discretizer.summary.loc[[str(discretizer.features[feature])]]

content
feature                     label                                     
Numerical('hours-per-week') (-inf, 1.00e+01]          (-inf, 1.00e+01]
                            (1.00e+01, 1.50e+01]  (1.00e+01, 1.50e+01]
                            (1.50e+01, 2.00e+01]  (1.50e+01, 2.00e+01]
                            (2.00e+01, 2.50e+01]  (2.00e+01, 2.50e+01]
                            (2.50e+01, 3.40e+01]  (2.50e+01, 3.40e+01]
                            (3.40e+01, 3.50e+01]  (3.40e+01, 3.50e+01]
                            (3.50e+01, 3.90e+01]  (3.50e+01, 3.90e+01]
                            (3.90e+01, 4.00e+01]  (3.90e+01, 4.00e+01]
                            (4.00e+01, 4.40e+01]  (4.00e+01, 4.40e+01]
                            (4.40e+01, 4.50e+01]  (4.40e+01, 4.50e+01]
                            (4.50e+01, 4.90e+01]  (4.50e+01, 4.90e+01]
                            (4.90e+01, 5.40e+01]  (4.90e+01, 5.40e+01]
                            (5.40e+01, 5.50e+01]  (5.40e+01, 5.50e+01]
                            (5.50e+01, 6.00e+01]  (5.50e+01, 6.00e+01]
                            (6.00e+01, inf)            (6.00e+01, inf)

In [15]:
# Not discretized distribution
stats = train_set[feature].value_counts(dropna=True, normalize=True)
print(f"Over-represented values of {feature}:\n", stats[stats >= min_freq])

Over-represented values of hours-per-week:
 hours-per-week
40    0.465232
50    0.087682
45    0.055947
60    0.045069
35    0.038978
20    0.038364
30    0.035088
55    0.021396
Name: proportion, dtype: float64


In [16]:
# Discretized distribution
disc_stats = train_set_processed[feature].value_counts(dropna=True, normalize=True)
print(f"Discretized distribution of {feature}:\n", disc_stats)

Discretized distribution of hours-per-week:
 hours-per-week
(3.90e+01, 4.00e+01]    0.465232
(4.90e+01, 5.40e+01]    0.094336
(4.40e+01, 4.50e+01]    0.055947
(2.50e+01, 3.40e+01]    0.051621
(5.50e+01, 6.00e+01]    0.049369
(1.50e+01, 2.00e+01]    0.048678
(3.40e+01, 3.50e+01]    0.038978
(6.00e+01, inf)         0.034551
(2.00e+01, 2.50e+01]    0.029586
(3.50e+01, 3.90e+01]    0.027820
(-inf, 1.00e+01]        0.022548
(5.40e+01, 5.50e+01]    0.021396
(4.50e+01, 4.90e+01]    0.021012
(1.00e+01, 1.50e+01]    0.020065
(4.00e+01, 4.40e+01]    0.018862
Name: proportion, dtype: float64


In [17]:
# values between 50 and 55 hours per week are under-represented
print(f"Observed data for {feature} values strictly between 50 and 55: {train_set.query(f'50<`{feature}`<55').shape[0] / len(train_set):.2%}")

Observed data for hours-per-week values strictly between 50 and 55: 0.67%


In [18]:
print(f"Target rate for {feature} values equal to 50: {train_set.query(f'`{feature}`==50')[target].mean():.2%}")
print(f"Target rate for {feature} values strictly between 50 and 55: {train_set.query(f'50<`{feature}`<55')[target].mean():.2%}")
print(f"Target rate for {feature} values equal to 55: {train_set.query(f'`{feature}`==55')[target].mean():.2%}")

Target rate for hours-per-week values equal to 50: 44.75%
Target rate for hours-per-week values strictly between 50 and 55: 30.00%
Target rate for hours-per-week values equal to 55: 47.13%


For quantitative discrete feature ``hours-per-week``:

* Some over-represented values have been identified: values ``20``, ``30``, ``35``, ``40``, ``45``, ``50``, ``55`` and ``60`` each represent more than 2.0% of observed data (more than ``min_freq=0.02``)
    * ``35``, ``40``, ``45`` and ``55`` are each kept as their own modality
    * ``20``, ``30``, ``50`` and ``60`` stay in separate modalities, each shared with neighbouring, less frequent values
* For instance, values strictly between ``50`` and ``55`` represent only 0.67% of observed data:
    * both of their neighbours, ``50`` and ``55``, are over-represented, so they are grouped with the neighbour whose target rate is the closest
    * their target rate (30.00%) is closer to the one of ``50`` (44.75%) than to the one of ``55`` (47.13%), hence the modality ``(4.90e+01, 5.40e+01]``
* The other values have been cut into quantiles

## Qualitative Categorical Feature

In [19]:
# Discretization Summary
feature = "workclass"
discretizer.summary.loc[[str(discretizer.features[feature])]]

content
feature                  label                                                            
Categorical('workclass') Never-worked, Without-pay  [Never-worked, Without-pay, __OTHER__]
                         Private                                                   Private
                         State-gov                                               State-gov
                         Self-emp-not-inc                                 Self-emp-not-inc
                         Local-gov                                               Local-gov
                         Federal-gov                                           Federal-gov
                         Self-emp-inc                                         Self-emp-inc

In [20]:
# Not discretized distribution
stats = train_set[feature].value_counts(dropna=True, normalize=True)
print(f"Over-represented values of {feature}:\n", stats[stats >= min_freq])
print(f"\nUnder-represented values of {feature}:\n", stats[stats < min_freq])

Over-represented values of workclass:
 workclass
Private             0.735597
Self-emp-not-inc    0.083123
Local-gov           0.069256
State-gov           0.042823
Self-emp-inc        0.037260
Federal-gov         0.031236
Name: proportion, dtype: float64

Under-represented values of workclass:
 workclass
Without-pay     0.000434
Never-worked    0.000271
Name: proportion, dtype: float64


In [21]:
# Discretized distribution
disc_stats = train_set_processed[feature].value_counts(dropna=False, normalize=True)
print(f"Discretized distribution of {feature}:\n", disc_stats)

Discretized distribution of workclass:
 workclass
Private                      0.693727
Self-emp-not-inc             0.078392
Local-gov                    0.065314
NaN                          0.056919
State-gov                    0.040386
Self-emp-inc                 0.035139
Federal-gov                  0.029458
Never-worked, Without-pay    0.000665
Name: proportion, dtype: float64


For qualitative categorical feature ``workclass``:

* Some over-represented categories have been identified:
    * categories ``"Private"``, ``"Self-emp-not-inc"``, ``"Local-gov"``, ``"State-gov"``, ``"Self-emp-inc"`` and ``"Federal-gov"`` each represent more than 2.0% of the non-missing values (more than ``min_freq=0.02``)
    * they are kept as their own modality
* Some under-represented categories have been identified:
    * categories ``"Without-pay"`` (0.04%) and ``"Never-worked"`` (0.03%) are much less frequent than ``min_freq=0.02``
    * they are grouped into the default value ``"__OTHER__"``, whose label lists them: ``"Never-worked, Without-pay"``
* Missing values are left as they are (5.7% of the rows are ``NaN`` after discretization). A ``Discretizer`` does not group them, while **Carvers** do by default.

## Qualitative Ordinal Feature

In [22]:
# Discretization Summary
feature = "education"
discretizer.summary.loc[[str(discretizer.features[feature])]]

content
feature              label                                                                
Ordinal('education') Preschool to 9th          [9th, 5th-6th, 1st-4th, Preschool, 7th-8th]
                     10th                                                             10th
                     11th to 12th                                             [12th, 11th]
                     HS-grad                                                       HS-grad
                     Some-college                                             Some-college
                     Assoc-voc                                                   Assoc-voc
                     Assoc-acdm                                                 Assoc-acdm
                     Bachelors                                                   Bachelors
                     Masters                                                       Masters
                     Prof-school to Doctorate                     [Doctorate, Prof-school]

In [23]:
# Not discretized distribution
stats = train_set[feature].value_counts(dropna=True, normalize=True)
print(f"Over-represented values of {feature}:\n", stats[stats >= min_freq])
print(f"\nUnder-represented values of {feature}:\n", stats[stats < min_freq])

Over-represented values of education:
 education
HS-grad         0.324751
Some-college    0.220920
Bachelors       0.164257
Masters         0.054820
Assoc-voc       0.041973
11th            0.037110
Assoc-acdm      0.032657
10th            0.028613
Name: proportion, dtype: float64

Under-represented values of education:
 education
7th-8th        0.019348
Prof-school    0.017045
9th            0.015407
12th           0.013488
Doctorate      0.012413
5th-6th        0.010212
1st-4th        0.005221
Preschool      0.001766
Name: proportion, dtype: float64


In [24]:
# Discretized distribution
disc_stats = train_set_processed[feature].value_counts(dropna=True, normalize=True)
print(f"Discretized distribution of {feature}:\n", disc_stats)

Discretized distribution of education:
 education
HS-grad                     0.324751
Some-college                0.220920
Bachelors                   0.164257
Masters                     0.054820
Preschool to 9th            0.051954
11th to 12th                0.050598
Assoc-voc                   0.041973
Assoc-acdm                  0.032657
Prof-school to Doctorate    0.029458
10th                        0.028613
Name: proportion, dtype: float64


For qualitative ordinal feature ``education``:

* Some over-represented categories have been identified:
    * categories ``"10th"``, ``"11th"``, ``"HS-grad"``, ``"Some-college"``, ``"Assoc-voc"``, ``"Assoc-acdm"``, ``"Bachelors"`` and ``"Masters"`` each represent more than 2.0% of observed data (more than ``min_freq=0.02``)
* Some under-represented categories have been identified:
    * categories ``"Preschool"``, ``"1st-4th"``, ``"5th-6th"``, ``"7th-8th"``, ``"9th"``, ``"12th"``, ``"Prof-school"`` and ``"Doctorate"`` each represent less than 2.0% of observed data
    * they are grouped with neighbouring categories, following the order given in ``ordinals``:
        - ``"Preschool"`` to ``"9th"`` form one modality (5.2% of observed data)
        - ``"12th"`` is grouped with ``"11th"``
        - ``"Doctorate"`` is grouped with ``"Prof-school"``
* Each label shows the range of categories it covers, for instance ``"Preschool to 9th"``

## Saving and Loading Discretizer
### Saving

All **Discretizers** can safely be stored as a .json file. Here we write it to the system's temporary folder.

In [25]:
import tempfile
from pathlib import Path

discretizer_file = Path(tempfile.gettempdir()) / "discretizer.json"

# save/load accept a str or a pathlib.Path
discretizer.save(discretizer_file)

### Loading

**Discretizers** can safely be loaded from a .json file.

In [26]:
discretizer = Discretizer.load(discretizer_file)

## Applying Discretizer

In [27]:
dev_set_processed = discretizer.transform(dev_set)
dev_set_processed[["capital-gain", "hours-per-week", "workclass", "education"]].head()

,capital-gain,hours-per-week,workclass,education
40342,"(-inf, 0.00e+00]","(3.90e+01, 4.00e+01]",Private,HS-grad
47680,"(-inf, 0.00e+00]","(1.50e+01, 2.00e+01]",Local-gov,HS-grad
524,"(-inf, 0.00e+00]","(4.90e+01, 5.40e+01]",Local-gov,Some-college
8508,"(0.00e+00, 3.41e+03]","(1.50e+01, 2.00e+01]",Private,HS-grad
31692,"(-inf, 0.00e+00]","(3.90e+01, 4.00e+01]",Local-gov,Assoc-acdm


## What's next?

* **Discretizers** are built into **Carvers**, which then search for the grouping of these modalities most associated with the target: see [Carvers Examples](../../../carvers_examples.rst).